#### 패키지 호출

In [ ]:
from com_fun import *
import geopandas as gpd

#### 상품권 할인율별 판매차이 분석

In [ ]:
merge_gift_user = pd.read_pickle('../P1_Data/02.Pretreatment/pkl/merge_gift_user.pkl')
merge_gift_user

In [ ]:
merge_gift_user02 = merge_gift_user.copy()
# 회원정보 탈퇴 삭제
merge_gift_user02 = merge_gift_user02[merge_gift_user02['회원상태'] != '탈퇴']

# 상품권 항목만 선택
# merge_gift_user02 = merge_gift_user02[merge_gift_user02['수단구분'] == '상품권']

# 가입연도 추가
merge_gift_user02['가입일자_연도'] = pd.to_datetime(merge_gift_user02['가입일자']).dt.year

# 구매금액 분포 추가
merge_gift_user02['만원단위'] = (merge_gift_user02['구매금액'] // 10000) * 10000
merge_gift_user02['구매금액_구간'] = merge_gift_user02['만원단위'].apply(lambda x: '20만원이상' if x >= 200000 else '1만원이하' if x == 0 else x)

# 할인율 추가
merge_gift_user02['할인율'] = merge_gift_user02['할인금액'] / merge_gift_user02['총금액']

# 필요없는 컬럼 제외
merge_gift_user02.drop(['지자체명', '구매_지급일자', '구매_지급시간', '가입일자','생년월일', '만원단위'], axis = 1, inplace = True)

# 정렬
merge_gift_user02 = merge_gift_user02[['구매_지급번호', '회원번호', '시도명', '시군구명', '주거래지역', '수단구분', 
                                       '구매_지급일', '구매_연도', '구매_월', '구매_일', '구매금액', '할인금액', '총금액', '구매금액_구간', '할인율',
                                       '회원상태', '가입일자_연도', '성별', '연령', '연령대']]

# 구매금액 0원 제외
merge_gift_user02 = merge_gift_user02[merge_gift_user02['구매금액'] != 0]

In [ ]:
# 성별 값 변환
merge_gift_user02['성별'] = merge_gift_user02['성별'].replace({1 : '남자', 2 : '여자'})

# 연령대 변환
merge_gift_user02['연령대'] = merge_gift_user02['연령대'].replace({0 : '10대 이하', 10 : '10대 이하', 20 : '20대', 30 : '30대', 40 : '40대', 50 : '50대', 60 : '60대 이상', 70 : '60대 이상', 80 : '60대 이상', 90 : '60대 이상', 100 : '60대 이상'})
merge_gift_user02.to_pickle('../P1_Data/02.Pretreatment/pkl/pre_merge_gift_user.pkl')

In [ ]:
# 지역기준 시각화 분석 셋 생성
view_df_buy02 = merge_gift_user02[['구매_연도', '구매_월', '시도명', '시군구명', '할인율', '성별', '연령대','구매_지급번호']].\
                groupby(['구매_연도', '구매_월', '시도명', '시군구명', '성별', '연령대', '할인율']).agg({'count'}).reset_index()
view_df_buy02.columns = view_df_buy02.columns.droplevel(1)
view_df_buy02.columns = ['구매_연도', '구매_월', '시도명', '시군구명', '성별', '연령대', '할인율', '구매건수_합계']
view_df_buy02.to_excel('../P3_Result/01.Excel/연도별_지역별_할인율_구매횟수.xlsx', index = False)
view_df_buy02

In [ ]:
# 지역기준 시각화 분석 셋 생성
view_df_buy02 = merge_gift_user02[['구매_연도', '구매_월', '시도명', '시군구명', '할인율','구매금액']].\
                groupby(['구매_연도', '구매_월', '시도명', '시군구명', '할인율']).agg({'sum'}).reset_index()
view_df_buy02.columns = view_df_buy02.columns.droplevel(1)
view_df_buy02.columns = ['구매_연도', '구매_월', '시도명', '시군구명', '할인율', '구매금액_합계']
view_df_buy02.to_excel('../P3_Result/01.Excel/연도별_지역별_할인율_구매금액.xlsx', index = False)
view_df_buy02

In [ ]:
# 지역별 할인율 및 구매횟수 확인
view_df_buy03 = view_df_buy02[['시도명', '시군구명', '할인율', '구매건수_합계']].groupby(['시도명', '시군구명', '할인율']).agg({'sum'}).reset_index()
view_df_buy03.columns = view_df_buy03.columns.droplevel(1)
view_df_buy03.columns = ['시도명', '시군구명', '할인율', '구매횟수']
view_df_buy03

In [ ]:
# 할인율별 구매건수 확인
view_df_buy03 = view_df_buy02[['할인율', '구매건수_합계']].groupby(['할인율']).agg({'sum'}).reset_index()
view_df_buy03.columns = view_df_buy03.columns.droplevel(1)
view_df_buy03.columns = ['할인율', '구매횟수']
view_df_buy03